In [4]:
import os
import zipfile
import re
import math
import pandas as pd
from collections import defaultdict, Counter
import gc

print("--- STARTING PIPELINE: BAWE & WORKSHOP ESSAYS ---")

# ==========================================
# SETUP: UNZIP EVERYTHING
# ==========================================
# UPDATED: File and Folder names as per your workshop setup
if not os.path.exists("bawe_corpus"):
    print("Unzipping BAWE corpus (BAWE_CORPUS_TXT.zip)...")
    with zipfile.ZipFile("BAWE_CORPUS_TXT.zip", 'r') as zip_ref:
        zip_ref.extractall("bawe_corpus")

if not os.path.exists("my_essays"):
    print("Unzipping student essays (Essays_for_Workshop.zip)...")
    with zipfile.ZipFile("Essays_for_Workshop.zip", 'r') as zip_ref:
        zip_ref.extractall("my_essays")

# Single, consistent regex tokenizer for the entire project
pattern = re.compile(r'[^\W\d_]+(?:-[^\W\d_]+)*', re.UNICODE)

# ==========================================
# PHASE 1: BUILD TRUE ALDF MASTER DICTIONARY
# ==========================================
print("\n--- PHASE 1: BUILDING BAWE MASTER DICTIONARY ---")
word_positions = defaultdict(list)
global_position = 0

print("Reading 6.5 million BAWE tokens...")
for root, dirs, files in os.walk("bawe_corpus"):
    for file in sorted(files):
        if file.endswith(".txt"):
            filepath = os.path.join(root, file)
            with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
                tokens = pattern.findall(f.read().lower())
                for token in tokens:
                    word_positions[token].append(global_position)
                    global_position += 1

TOTAL_TOKENS = global_position
print(f"Total tokens processed: {TOTAL_TOKENS:,}")

# Clear text from memory before heavy math
gc.collect()

print("Calculating TRUE ALDF (Sketch Engine Linear Formula)...")
ref_dict = {} # Stores the ALDF scores in memory for instant scoring
dict_results = []

for token, positions in word_positions.items():
    freq = len(positions)
    if freq >= 5:
        # Calculate gaps (Strictly Linear Model)
        gaps = []
        for i in range(1, freq):
            gaps.append(positions[i] - positions[i-1])

        # STEP 1: Calculate ALD (Average Logarithmic Distance)
        # Formula: (1/N) * sum(d * log2(d))
        # Note: Normalization is over TOTAL_TOKENS (N) as per Sketch Engine
        ald_sum = sum(d * math.log2(d) for d in gaps if d > 0)
        ald = ald_sum / TOTAL_TOKENS

        # STEP 2: Convert ALD to ALDF (Sketch Engine's Effective Frequency)
        # Formula: N / (2^ALD)
        aldf_score = round(TOTAL_TOKENS / (2 ** ald), 2)

        ref_dict[token] = aldf_score # Fast memory lookup

        dict_results.append({
            'Token': token,
            'Freq': freq,
            'ALDF': aldf_score
        })

# Save dictionary to CSV for your records
pd.DataFrame(dict_results).sort_values(by='Freq', ascending=False).to_csv("BAWE_TRUE_ALDF.csv", index=False)
print("Master Dictionary saved as 'BAWE_TRUE_ALDF.csv'.")

# Clear heavy memory variables before Phase 2
del word_positions
del dict_results
gc.collect()


# ==========================================
# PHASE 2: SCORE STUDENT ESSAYS
# ==========================================
print("\n--- PHASE 2: SCORING WORKSHOP ESSAYS ---")

function_words = set([
    "the", "a", "an", "some", "any", "every", "all", "both", "neither", "either", "no", "each", "another",
    "i", "me", "my", "mine", "we", "us", "our", "ours", "you", "your", "yours",
    "he", "him", "his", "she", "her", "hers", "it", "its", "they", "them", "their", "theirs",
    "this", "that", "these", "those",
    "in", "on", "at", "by", "for", "with", "about", "against", "between", "into", "through",
    "during", "before", "after", "above", "below", "to", "from", "up", "down", "of", "over",
    "under", "upon", "within", "without", "among", "beyond", "behind",
    "and", "but", "or", "yet", "so", "nor", "although", "because", "since", "unless", "while", "if", "than", "as", "until",
    "is", "am", "are", "was", "were", "be", "being", "been",
    "has", "have", "had", "do", "does", "did",
    "will", "would", "shall", "should", "can", "could", "may", "might", "must",
    "here", "there", "when", "where", "why", "how", "then", "once", "again", "further", "too", "very", "not"
])

results = []
essays_dir = "my_essays" # This points to the unzipped folder from Essays_for_Workshop.zip

for root, dirs, files in os.walk(essays_dir):
    for filename in files:
        if filename.endswith(".txt"):
            filepath = os.path.join(root, filename)

            with open(filepath, 'r', encoding='utf-8', errors='ignore') as file:
                text = file.read().lower()
                tokens = pattern.findall(text)
                token_counts = Counter(tokens)

                all_total, all_count = 0, 0
                func_total, func_count = 0, 0
                content_total, content_count = 0, 0

                for token, freq in token_counts.items():
                    if token in ref_dict:
                        score = ref_dict[token]
                        all_total += freq * score
                        all_count += freq

                        if token in function_words:
                            func_total += freq * score
                            func_count += freq
                        else:
                            content_total += freq * score
                            content_count += freq

                mean_all = all_total / all_count if all_count else 0
                mean_func = func_total / func_count if func_count else 0
                mean_content = content_total / content_count if content_count else 0
                coverage = (all_count / len(tokens)) * 100 if tokens else 0

                results.append({
                    "Essay": filename,
                    "Total_Tokens": len(tokens),
                    "BAWE_Coverage_Percent": round(coverage, 2),
                    "BAWE_ALDF_AW": round(mean_all, 4),
                    "BAWE_ALDF_Function": round(mean_func, 4),
                    "BAWE_ALDF_Content": round(mean_content, 4)
                })

pd.DataFrame(results).to_csv("ALDF_Scores.csv", index=False)

print("\n--- PIPELINE COMPLETE ---")
print("SUCCESS! Final scores saved to 'ALDF_Scores.csv'.")

--- STARTING PIPELINE: BAWE & WORKSHOP ESSAYS ---

--- PHASE 1: BUILDING BAWE MASTER DICTIONARY ---
Reading 6.5 million BAWE tokens...
Total tokens processed: 6,648,685
Calculating TRUE ALDF (Sketch Engine Linear Formula)...
Master Dictionary saved as 'BAWE_TRUE_ALDF.csv'.

--- PHASE 2: SCORING WORKSHOP ESSAYS ---

--- PIPELINE COMPLETE ---
SUCCESS! Final scores saved to 'ALDF_Scores.csv'.
